# Day 3: Functions

### The mindset first.
    In mathematics, a function is a mapping — input goes in, output comes out, nothing else changes. Python functions can work the same way, and when they do, your code becomes predictable, testable, and composable. A function that modifies something outside itself is harder to debug and harder to reuse.
    In data science, you are constantly building pipelines: raw data → clean data → features → model → predictions. Each arrow is a function. If your functions are well-designed, the pipeline is easy to reason about. If they're not, bugs hide in the connections.

### Concept 1: Function basics and return values

In [3]:
def square(x):
    return x**2
# Functions can return multiple values as a tuple
def min_max(data):
    return min(data), max(data) # returns a tuple
low,high= min_max([3, 1, 4, 1, 5, 9]) # unpacking
print(low,high)

1 9


### A function without a return statement returns None. This is a common silent bug — you call a function expecting a value and get None instead.

# Concept 2: Default arguments

In [2]:
def normalize(data, method='minmax'):
    if method =="minmax":
        mn, mx = min(data), max(data)
        return [(x-mn)/(mx-mn) for x in data]
    elif method == "zscore":
        mean = sum(data)/len(data)
        std= (sum((x-mean)**2 for x in data)/len(data))**0.5
        return [ (x-mean)/std for x in data]
print(normalize([1, 2, 3, 4, 5]))              # uses minmax by default
print(normalize([1, 2, 3, 4, 5], "zscore"))    # overrides default

[0.0, 0.25, 0.5, 0.75, 1.0]
[-1.414213562373095, -0.7071067811865475, 0.0, 0.7071067811865475, 1.414213562373095]


# Concept 3: *args and **kwargs

### These let you write functions that accept a variable number of arguments.

In [1]:
# *args — variable positional arguments, received as a tuple
def mean(*args):
    return sum(args) / len(args)

mean(1, 2, 3)          # 2.0
mean(10, 20, 30, 40)   # 25.0

# **kwargs — variable keyword arguments, received as a dict
def build_model_config(**kwargs):
    config = {"learning_rate": 0.01, "epochs": 100}   # defaults
    config.update(kwargs)                               # override with provided
    return config

build_model_config(epochs=200, batch_size=32)
# {"learning_rate": 0.01, "epochs": 200, "batch_size": 32}

{'learning_rate': 0.01, 'epochs': 200, 'batch_size': 32}

# Concept 4: Scope

In [6]:
threshold = 0.5   # global scope
def classify(prob):
    threshold = 0.8   # local scope — does NOT change the global
    return "positive" if prob >= threshold else "negative"
classify(0.7)
print(threshold)   # still 0.5 — global unchanged

0.5


### Variables inside a function are local — they are created when the function runs and destroyed when it returns. They never affect the outside world unless you explicitly return them. If you need to read a global variable inside a function, Python finds it automatically. If you need to modify it — which you rarely should — use the global keyword. But modifying globals is a code smell. Return values instead.

# Concept 5: Functions as first-class objects

### This is where Python gets powerful for data work. Functions can be passed as arguments, stored in lists, and returned from other functions.

In [9]:
def apply_transform(data, func):
    return [func(x) for x in data]
def square(x): return x ** 2
def log(x): return x ** 0.5
data = [1, 4, 9, 16, 25]
print(apply_transform(data, square))  
print(apply_transform(data, log))     

[1, 16, 81, 256, 625]
[1.0, 2.0, 3.0, 4.0, 5.0]


### This is the foundation of how pandas' .apply() works — you pass a function and it applies it to every row or column.

# Concept 6: Lambda functions

In [10]:
square = lambda x: x ** 2
print(square(5))   # 25

# Most useful inline — passing a function without naming it
data = [("age", 0.8), ("bmi", 0.3), ("glucose", 0.9)]
sorted_data = sorted(data, key=lambda pair: pair[1], reverse=True)
# [("glucose", 0.9), ("age", 0.8), ("bmi", 0.3)]

25


# Concept 7: Closures

In [15]:
def make_scaler(min_val, max_val):
    def scale(x):
        return (x - min_val) / (max_val - min_val)
    return scale   # returns the function itself
# Train your scaler on training data
scaler = make_scaler(10, 50)
# Apply to any data later — min_val and max_val are remembered
print(scaler(10))   # 0.0
print(scaler(30))   # 0.5
print(scaler(50))   # 1.0

0.0
0.5
1.0


### scale remembers min_val and max_val even after make_scaler has finished running. This is exactly how sklearn's MinMaxScaler works conceptually — fit on training data, apply to test data.

# Simpified version

In [18]:
def make_scaler(x, min_val, max_val):
    return (x-min_val)/(max_val-min_val)
print(make_scaler(10,10,50))
print(make_scaler(30,10,50))
print(make_scaler(50,10,50))

0.0
0.5
1.0


# Practice Problems

### Problem 1 — Default arguments + return values:
    Write a function summarize(data, percentiles=None) where:
    percentiles defaults to [25, 50, 75] — use the None pattern
    Returns a dictionary with "mean", "std", and "percentiles" as a nested dict {25: value, 50: value, 75: value}
    To compute a percentile: sort the data, then index at int(p/100 * len(data))
    Compute std from scratch using a loop
    Test with [10, 20, 30, 40, 50, 60, 70, 80, 90, 100].

### Problem 2 — *args and kwargs:
    Write a function pipeline(data, *transforms) that applies a sequence of transform functions to the data in order, passing the output of each as input to the next.
    Then write three simple transform functions: remove_negatives, square_values, and normalize (min-max). Chain them together.
    Test with [-2, 3, -1, 4, 2, 5].

### Problem 3 — First-class functions + lambda:
    Write a function evaluate_metrics(actual, predicted, *metrics) where each metric is a function that takes (actual, predicted) and returns a float.
    Then define these three metric functions using def or lambda:
    accuracy — proportion of matching values
    precision — true positives / (true positives + false positives)
    recall — true positives / (true positives + false negatives)
    Call evaluate_metrics with all three and return a dict of {metric_name: value}. Use func.__name__ to get the function's name as the key.
    Test with:
    actual    = [1, 0, 1, 1, 0, 1, 0, 0]
    predicted = [1, 0, 0, 1, 0, 1, 1, 0]

### Problem 4 — Closures:
    Write a function make_standardizer(data) that:
    Computes mean and std from data (the training set)
    Returns a function standardize(x) that applies z-score scaling: (x - mean) / std
    The returned function should also have a describe() method — actually, just make make_standardizer return a tuple of (standardize_func, {"mean": mean, "std": std})
    Then simulate a train/test scenario:
    train = [10, 20, 30, 40, 50]
    test  = [15, 25, 35]
    standardizer, params = make_standardizer(train)
    print(params)
    print([standardizer(x) for x in test])